## Import Library

In [2]:
import json
import pandas as pd
import re
import yaml

In [4]:
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

## Load Data JSON

In [5]:
import os

raw_dir = config["data"]["interim_dir"]

modsec_file = os.path.join("..", raw_dir, "modsec_audit.json")

with open(modsec_file) as f:
    data = json.load(f)

len(data)

20008

In [6]:
df = pd.json_normalize(data)

df.head()

,id,sections.A-audit_header,sections.B-request_headers,sections.D-intermediate_body,sections.F-response_headers,sections.H-audit_messages,sections.I-additional_data,sections.J-trailer,sections.Z-postlude,sections.E-response_body
0,XD10tCwH,[05/May/2025:09:08:24 +0000] 174643610420.9220...,OPTIONS /api/v1/auths/login HTTP/1.1\nCf-Conne...,,"HTTP/1.1 204\nServer: nginx/1.26.3\nDate: Mon,...","ModSecurity: Warning. Matched ""Operator `Pm' w...",,,,NaN
1,sRNMSqLX,[05/May/2025:09:08:25 +0000] 17464361050.85987...,POST /api/v1/auths/login HTTP/1.1\nCf-Connecti...,,"HTTP/1.1 401\nServer: nginx/1.26.3\nDate: Mon,...","ModSecurity: Warning. Matched ""Operator `Pm' w...",,,,NaN
2,wWRXeQtO,[05/May/2025:09:09:36 +0000] 174643617613.6103...,OPTIONS /api/v1/auths/login HTTP/1.1\nCf-Conne...,,"HTTP/1.1 204\nServer: nginx/1.26.3\nDate: Mon,...","ModSecurity: Warning. Matched ""Operator `Pm' w...",,,,NaN
3,bg2mL7Uq,[05/May/2025:09:09:37 +0000] 174643617778.2575...,POST /api/v1/auths/login HTTP/1.1\nCf-Connecti...,,"HTTP/1.1 200\nServer: nginx/1.26.3\nDate: Mon,...","ModSecurity: Warning. Matched ""Operator `Pm' w...",,,,NaN
4,KTqEfmTV,[05/May/2025:09:09:37 +0000] 174643617764.8021...,GET /api/v1/books?search=&page=1&size=9 HTTP/1...,,"HTTP/1.1 200\nServer: nginx/1.26.3\nDate: Mon,...","ModSecurity: Warning. Matched ""Operator `Pm' w...",,,,NaN


In [7]:
def parse_request(request_text):
    first_line = request_text.split("\n")[0]

    parts = first_line.split(" ")

    if len(parts) >= 2:
        return parts[0], parts[1]

    return None, None


df["method"], df["endpoint"] = zip(*df["sections.B-request_headers"].apply(parse_request))

In [8]:
def extract_ip(headers):

    match = re.search(r"X-Forwarded-For:\s*(.*)", headers)

    if match:
        return match.group(1)

    return None

df["ip"] = df["sections.B-request_headers"].apply(extract_ip)

In [9]:
def extract_status(resp):

    match = re.search(r"HTTP\/1\.1\s+(\d+)", resp)

    if match:
        return int(match.group(1))

    return None


df["status"] = df["sections.F-response_headers"].apply(extract_status)

In [10]:
def extract_rule(msg):

    match = re.search(r'\[id \"(\d+)\"\]', msg)

    if match:
        return int(match.group(1))

    return None

df["rule_id"] = df["sections.H-audit_messages"].apply(extract_rule)

In [13]:
def extract_attack(msg):

    match = re.search(r'\[msg \"(.*?)\"\]', msg)

    if match:
        return match.group(1)

    return None

df["attack_type"] = df["sections.H-audit_messages"].apply(extract_attack)

In [14]:
df["attack"] = 1

In [15]:
modsec_df = df[[
    "ip",
    "method",
    "endpoint",
    "status",
    "rule_id",
    "attack_type",
    "attack"
]]

modsec_df.head()

,ip,method,endpoint,status,rule_id,attack_type,attack
0,175.45.191.10,OPTIONS,/api/v1/auths/login,204,10008.0,Midtrans SSRF Detected: Missing signature_key,1
1,175.45.191.10,POST,/api/v1/auths/login,401,10008.0,Midtrans SSRF Detected: Missing signature_key,1
2,175.45.191.15,OPTIONS,/api/v1/auths/login,204,10008.0,Midtrans SSRF Detected: Missing signature_key,1
3,175.45.191.10,POST,/api/v1/auths/login,200,10008.0,Midtrans SSRF Detected: Missing signature_key,1
4,175.45.191.10,GET,/api/v1/books?search=&page=1&size=9,200,10008.0,Midtrans SSRF Detected: Missing signature_key,1


In [16]:
modsec_df["attack_type"].value_counts()

attack_type
SQL Injection Detected                           17739
Midtrans SSRF Detected: Missing signature_key     2181
Possible XSS attack detected in request body        22
Name: count, dtype: int64

In [17]:
modsec_df["endpoint"].value_counts().head(10)

endpoint
/api/v1/auths/login                        17709
/api/v1/books?search=&page=1&size=9           90
/?token=Kp5rLtCf5Hrs                          36
/@vite/client                                 34
/src/main.tsx                                 34
/@react-refresh                               34
/src/App.tsx                                  34
/node_modules/vite/dist/client/env.mjs        34
/src/pages/admin/index.tsx                    34
/src/pages/admin/books/create/index.tsx       34
Name: count, dtype: int64

In [18]:
modsec_df["ip"].value_counts().head(10)

ip
175.45.188.252     18155
175.45.191.19        536
175.45.191.10        325
175.45.191.11        231
172.236.213.119      198
175.45.191.12         97
175.45.191.9          93
175.45.191.15         92
175.45.191.8          65
175.45.191.13         46
Name: count, dtype: int64

In [19]:
# raw_dir = config["data"]["processed_dir"]
#
# modsec_file_processed = os.path.join("..", raw_dir, "modsec_processed.csv")
#
# modsec_df.to_csv(modsec_file_processed, index=False)

In [20]:
modsec_df["endpoint_length"] = modsec_df["endpoint"].str.len()

modsec_df["param_count"] = modsec_df["endpoint"].str.count("&")

modsec_df["has_query"] = modsec_df["endpoint"].str.contains("\?").astype(int)

modsec_df["attack"] = 1

In [21]:
modsec_df = modsec_df[
    [
        "ip",
        "method",
        "endpoint",
        "status",
        "endpoint_length",
        "param_count",
        "has_query",
        "attack"
    ]
]

In [22]:
raw_dir = config["data"]["processed_dir"]

modsec_file_processed = os.path.join("..", raw_dir, "modsec_parsed.csv")

modsec_df.to_csv(modsec_file_processed, index=False)